[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/21_dimensionality_reduction/01_pca_from_scratch.ipynb)

# Bài 01 — PCA from scratch

**🎯 Mục tiêu**
- Trực giác: tìm các **hướng** mà dữ liệu biến thiên nhiều nhất; chiếu dữ liệu lên vài hướng đầu.
- Hai góc nhìn tương đương: **tối đa phương sai** ↔ **tối thiểu sai số tái tạo**.
- Cài đặt PCA bằng **trị riêng của ma trận hiệp phương sai** và bằng **SVD**; đối chiếu sklearn.
- Explained variance ratio, chiếu (transform) và tái tạo (inverse transform).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/21_dimensionality_reduction

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

from ml_course.algorithms import PCAScratch
from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Trực giác 2D

Dữ liệu 2D tương quan mạnh: phần lớn biến thiên nằm dọc **một** hướng. Nếu chỉ giữ toạ độ dọc hướng đó (1 số thay vì 2), ta mất rất ít thông tin.

In [ ]:
n = 300
t = rng.normal(0, 2, n)
X = np.c_[t + rng.normal(0, 0.5, n), 0.6 * t + rng.normal(0, 0.5, n)] + [3, 1]
Xc = X - X.mean(0)
C = Xc.T @ Xc / (n - 1)                                  # ma trận hiệp phương sai 2×2
eigvals, eigvecs = np.linalg.eigh(C)                     # eigh: cho ma trận đối xứng, trị riêng tăng dần
order = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[order], eigvecs[:, order]
print("trị riêng (phương sai dọc mỗi hướng):", eigvals.round(3))
print("tỉ lệ phương sai giải thích:", (eigvals / eigvals.sum()).round(3))

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(*X.T, s=10, alpha=0.5)
for val, vec, color in zip(eigvals, eigvecs.T, ["r", "g"]):
    ax.annotate("", X.mean(0) + 2 * np.sqrt(val) * vec, X.mean(0), arrowprops=dict(color=color, width=2))
ax.set_aspect("equal")
ax.set_title("Hướng thành phần chính (đỏ: PC1, xanh: PC2), độ dài ∝ độ lệch chuẩn")
plt.show()

## 2. Toán học

Dữ liệu đã **trừ trung bình** $X_c$ ($n \times d$). Tìm vector đơn vị $\mathbf w$ sao cho phương sai của hình chiếu $X_c\mathbf w$ lớn nhất:

$$\max_{\lVert\mathbf w\rVert = 1}\ \mathbf w^\top C\,\mathbf w, \qquad C = \frac{1}{n - 1}X_c^\top X_c$$

Lagrange → $C\mathbf w = \lambda\mathbf w$: $\mathbf w$ là **vector riêng** của $C$, phương sai đạt được bằng **trị riêng** $\lambda$. PC1 = vector riêng có $\lambda$ lớn nhất; PC2 = vector riêng lớn thứ hai (trực giao với PC1)...

**Góc nhìn tái tạo:** chiếu lên $k$ thành phần đầu rồi tái tạo $\hat X = X_c W_k W_k^\top$ — PCA tối thiểu $\lVert X_c - \hat X\rVert_F^2$, và sai số còn lại = tổng các trị riêng bị bỏ.

**Qua SVD** (ổn định số học hơn, không cần lập $C$): $X_c = U\Sigma V^\top$ → các cột của $V$ là thành phần chính, $\lambda_j = \sigma_j^2/(n - 1)$.

In [ ]:
def pca_eig(X, k):
    mean = X.mean(0)
    Xc = X - mean
    vals, vecs = np.linalg.eigh(Xc.T @ Xc / (len(X) - 1))
    order = np.argsort(vals)[::-1][:k]
    return mean, vecs[:, order].T, vals[order]


def pca_svd(X, k):
    mean = X.mean(0)
    _, S, Vt = np.linalg.svd(X - mean, full_matrices=False)
    return mean, Vt[:k], S[:k] ** 2 / (len(X) - 1)


X5 = rng.normal(size=(500, 5)) @ rng.normal(size=(5, 5))       # 5 chiều tương quan
_, comp_eig, var_eig = pca_eig(X5, 3)
_, comp_svd, var_svd = pca_svd(X5, 3)
sk = PCA(3).fit(X5)
print("phương sai (eig):", var_eig.round(4))
print("phương sai (SVD):", var_svd.round(4))
print("sklearn         :", sk.explained_variance_.round(4))
# vector riêng chỉ xác định đến dấu ± → so sánh |tích vô hướng|
print("cùng hướng với sklearn:", np.allclose(np.abs((comp_svd * sk.components_).sum(1)), 1))

## 3. Chiếu và tái tạo

In [ ]:
mean, W, var = pca_svd(X, 1)
Z = (X - mean) @ W.T                     # (n, 1) — toạ độ trên PC1
X_hat = Z @ W + mean                     # tái tạo về 2D
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(*X.T, s=10, alpha=0.4, label="gốc")
ax.scatter(*X_hat.T, s=10, color="r", label="tái tạo từ 1 thành phần")
for i in range(0, n, 15):
    ax.plot([X[i, 0], X_hat[i, 0]], [X[i, 1], X_hat[i, 1]], "k-", lw=0.5)
ax.set_aspect("equal")
ax.legend()
ax.set_title("Sai số tái tạo = khoảng cách vuông góc tới đường PC1")
plt.show()
mse = np.mean(np.sum((X - X_hat) ** 2, axis=1))
print(f"sai số tái tạo trung bình = {mse:.4f} | trị riêng bị bỏ (PC2) × (n−1)/n = {eigvals[1] * (n - 1) / n:.4f}")

Sai số tái tạo bằng đúng phương sai của thành phần bị bỏ — xác nhận hai góc nhìn tương đương.

> 💡 PCA ≠ Linear Regression: hồi quy tối thiểu sai số **dọc trục y**, PCA tối thiểu khoảng cách **vuông góc** tới đường thẳng, và đối xứng giữa các biến (không có biến "mục tiêu").

## 4. Explained variance và chọn số thành phần

In [ ]:
digits = load_digits()
pca_full = PCA().fit(digits.data)
cum = np.cumsum(pca_full.explained_variance_ratio_)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(range(1, 31), pca_full.explained_variance_ratio_[:30])
axes[0].set_title("Scree plot (30 thành phần đầu, digits 64 chiều)")
axes[1].plot(range(1, 65), cum)
for thr in [0.8, 0.9, 0.95]:
    k = int(np.searchsorted(cum, thr) + 1)
    axes[1].axhline(thr, color="gray", ls=":")
    axes[1].annotate(f"{thr:.0%}: {k} thành phần", (k, thr), textcoords="offset points", xytext=(5, -12))
axes[1].set_xlabel("số thành phần")
axes[1].set_ylabel("phương sai tích luỹ")
plt.show()

## 5. Phiên bản tham chiếu

In [ ]:
ref = PCAScratch(n_components=0.95).fit(digits.data)
print(f"PCAScratch giữ {ref.n_components_} thành phần cho 95% phương sai | sklearn: {PCA(0.95).fit(digits.data).n_components_}")

## 🧠 Kiểm tra nhanh

1. Vì sao phải trừ trung bình trước PCA?
2. PC1 là vector riêng ứng với trị riêng lớn nhất hay nhỏ nhất của ma trận hiệp phương sai? Vì sao?
3. Ba trị riêng 6, 3, 1. Tỉ lệ phương sai giải thích của 2 thành phần đầu?
4. PCA khác Linear Regression ở đâu khi "vẽ đường thẳng qua dữ liệu"?

## 📌 Tóm tắt

- PCA tìm hướng phương sai lớn nhất = vector riêng của ma trận hiệp phương sai; tương đương tối thiểu sai số tái tạo.
- Tính qua SVD của dữ liệu đã trừ trung bình; $\lambda = \sigma^2/(n - 1)$.
- Explained variance ratio giúp chọn số thành phần.

## 📚 Tài liệu tham khảo

- Shlens (2014) — *A Tutorial on Principal Component Analysis*
- Vũ Hữu Tiệp — [Principal Component Analysis](https://machinelearningcoban.com/2017/06/15/pca/)
- ESL §14.5